# Renk sınıflandırma ön denemesi (TÜBİTAK 2209-A)

Bu defterde kendi çektiğin fotoğraflardan pikselleri 4 renge (kırmızı, yeşil, mavi, sarı) ayıran
küçük bir yapay sinir ağı (tek katmanlı doğrusal sınıflandırıcı) eğiteceğiz, test edeceğiz ve
breadboard'daki donanımın göreceği 4 bitlik haline indirip doğruluk kaybını ölçeceğiz.

Hücreleri yukarıdan aşağıya sırayla çalıştır (Shift + Enter).

## 1. Hazırlık
Kütüphaneleri yüklüyoruz ve sonuçları tekrarlanabilir yapmak için rastgele sayı tohumunu sabitliyoruz.

In [ ]:
import os, glob, zipfile, re
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

np.random.seed(0)
RENKLER = ["kirmizi", "yesil", "mavi", "sari"]   # sınıf sırası: 0, 1, 2, 3
# Dosya adlarında İngilizce veya Türkçe karakterli adlar da kabul edilir (red_1.png, kırmızı_1.jpg ...)
ESANLAMLI = {"red": "kirmizi", "kırmızı": "kirmizi", "green": "yesil", "yeşil": "yesil",
             "blue": "mavi", "yellow": "sari", "sarı": "sari"}
def renk_adi(s):
    s = s.lower()
    return ESANLAMLI.get(s, s)
print("Hazır.")

## 2. Fotoğrafları yükle
Eğitim görselleri: her renkten en az 5 tane, adı `renk_sayı` olsun. Türkçe veya İngilizce olabilir:
`kirmizi_1.jpg` ya da `red_1.png`, `green_2.png`, `blue_3.png`, `yellow_4.png` ...

Test sahneleri: 5×5 karışık renkli görseller, adı `sahne` veya `test` ile başlasın ve en çok bulunan renkle bitsin:
`sahne_01_sari.png`, `test_02_blue.png` ...

Hepsini (klasörlü ya da klasörsüz) tek bir `foto.zip` olarak sıkıştır. Aşağıdaki hücreyi çalıştırınca çıkan **Dosya seç** düğmesiyle yükle.

In [ ]:
try:
    from google.colab import files
    yuklenen = files.upload()            # foto.zip'i seç
    zip_adi = list(yuklenen.keys())[0]
except ImportError:                       # Colab dışında çalışıyorsan zip'i bu klasöre koy
    zip_adi = "foto.zip"
with zipfile.ZipFile(zip_adi) as z:
    z.extractall("veri")

tum = [d for d in glob.glob("veri/**/*.*", recursive=True)
       if d.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".gif", ".webp")) and "__MACOSX" not in d]
ad = lambda d: os.path.splitext(os.path.basename(d))[0]
# Eğitim: adı "renk_sayı" olanlar (red_1, kirmizi_3). Sahne: adı "sahne" veya "test" ile başlayıp baskın renkle bitenler.
egitim_dosyalari = sorted(d for d in tum if renk_adi(ad(d).split("_")[0]) in RENKLER)
sahne_dosyalari = sorted(d for d in tum if ad(d).lower().startswith(("sahne", "test"))
                         and renk_adi(ad(d).split("_")[-1]) in RENKLER)
print(len(egitim_dosyalari), "eğitim fotoğrafı,", len(sahne_dosyalari), "sahne fotoğrafı bulundu.")

## 3. Fotoğraflardan piksel verisi çıkar
Her fotoğrafı 100×100'e küçültüyoruz ve kenarlarda masa ya da gölge olabileceği için sadece
ortadaki 60×60'lık bölgeyi alıyoruz. Her piksel bir örnek: girdi (R, G, B), etiket = dosya adındaki renk.

**Eğitim/test ayrımı fotoğraf bazında:** her rengin son fotoğrafı teste ayrılır, model onu hiç görmez.
Aynı fotoğrafın pikselleri hem eğitimde hem testte olsaydı test çok kolay olur, sonuç şişerdi.

In [ ]:
def pikselleri_al(yol):
    img = Image.open(yol).convert("RGB")
    if max(img.size) <= 64:                        # 5x5, 20x20 gibi küçük görseller: bütün pikseller
        return np.asarray(img).reshape(-1, 3).astype(float)
    img = img.resize((100, 100))
    dizi = np.asarray(img)[20:80, 20:80]          # fotoğraf: ortadaki 60x60
    return dizi.reshape(-1, 3).astype(float)       # 3600 piksel x (R,G,B)

X_egitim, y_egitim, X_test, y_test = [], [], [], []
for etiket, renk in enumerate(RENKLER):
    renk_dosyalari = [d for d in egitim_dosyalari if renk_adi(ad(d).split("_")[0]) == renk]
    renk_dosyalari.sort(key=lambda d: int(re.findall(r"\d+", os.path.basename(d))[0]))
    if len(renk_dosyalari) < 2:
        raise ValueError(f"'{renk}' için en az 2 fotoğraf gerekli, {len(renk_dosyalari)} bulundu.")
    for i, d in enumerate(renk_dosyalari):
        p = pikselleri_al(d)
        if i == len(renk_dosyalari) - 1:           # son fotoğraf -> test
            X_test.append(p); y_test += [etiket] * len(p)
        else:
            X_egitim.append(p); y_egitim += [etiket] * len(p)
    print(f"{renk}: {len(renk_dosyalari)-1} eğitim, 1 test fotoğrafı")

X_egitim, X_test = np.vstack(X_egitim), np.vstack(X_test)
y_egitim, y_test = np.array(y_egitim), np.array(y_test)
print("Eğitim pikseli:", len(X_egitim), " Test pikseli:", len(X_test))
farkli = len(np.unique(X_egitim, axis=0))
print("Eğitimdeki farklı renk sayısı:", farkli)
if farkli < 40:
    print("UYARI: Eğitim verisinde çok az farklı renk var. Model ezberler ve test doğruluğu gerçekte olduğundan",
          "yüksek çıkar. Farklı tonlar (koyu, açık, gölgeli) ya da gerçek fotoğraflar ekle.")

## 4. Modeli eğit (normal, kayan noktalı)
Her renk sınıfı için bir puan hesaplanır: `puan_k = w_k1·R + w_k2·G + w_k3·B + b_k`.
En yüksek puanlı sınıf pikselin rengidir. Eğitim, bu 4×3 ağırlığı ve 4 sapmayı (b) veriden öğrenir.
Girdileri 0–1 aralığına bölüyoruz (R/255) ki eğitim düzgün çalışsın.

In [ ]:
model = LogisticRegression(max_iter=2000)
model.fit(X_egitim / 255, y_egitim)
W = model.coef_          # 4 x 3 ağırlık
b = model.intercept_     # 4 sapma

tahmin = model.predict(X_test / 255)
dogruluk_float = (tahmin == y_test).mean()
print(f"Test doğruluğu (float): %{100*dogruluk_float:.1f}")
print("Ağırlıklar (satır = renk, sütun = R, G, B):\n", np.round(W, 2))

## 5. Hangi renkler karışıyor? (karışıklık matrisi)
Satır: gerçek renk, sütun: modelin dediği renk. Köşegen dışındaki sayılar hatalardır.

In [ ]:
cm = confusion_matrix(y_test, tahmin)
plt.imshow(cm, cmap="Blues")
plt.xticks(range(4), RENKLER); plt.yticks(range(4), RENKLER)
plt.xlabel("Tahmin"); plt.ylabel("Gerçek")
for i in range(4):
    for j in range(4):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.title("Karışıklık matrisi (float)"); plt.show()

## 6. 4 bite indirme (nicemleme)
Breadboard'da pikseller 4 bit (0–15), ağırlıklar işaretli 4 bit (−8…+7) olacak.
- Piksel: `R4 = R // 16` (255 → 15)
- Ağırlık: hepsi aynı ölçekle çarpılıp en büyüğü 7 olacak şekilde yuvarlanır.
- Sapma (b) akümülatöre başlangıç değeri olarak yüklenir, o yüzden 4 bite sığmak zorunda değil.

Sonra modeli tam sayılarla, yani donanımın yapacağı gibi çalıştırıp doğruluğu yeniden ölçüyoruz.

In [ ]:
def nicemle(W, b, bit):
    en_buyuk = 2**(bit - 1) - 1                    # 4 bit için 7
    olcek = en_buyuk / np.abs(W).max()
    Wq = np.clip(np.round(W * olcek), -en_buyuk - 1, en_buyuk).astype(int)
    # R/255 yerine R4 = R // 2^(8-bit) kullanıldığı için sapmayı aynı ölçeğe taşıyoruz.
    # // aşağı yuvarladığı için her R4, aslında R4 + 0.5 kadarlık bir değeri temsil eder;
    # bu yarım adımın puana etkisini (0.5 · ağırlıklar toplamı) sapmaya ekleyerek düzeltiyoruz.
    bq = np.round(olcek * (b * 2**bit + 0.5 * W.sum(axis=1))).astype(int)
    return Wq, bq

def piksel_nicemle(X, bit):
    return (X // 2**(8 - bit)).astype(int)

Wq, bq = nicemle(W, b, 4)
Xq_test = piksel_nicemle(X_test, 4)
puanlar = Xq_test @ Wq.T + bq                       # her piksel için 4 tam sayı puan
tahmin_q = puanlar.argmax(axis=1)
dogruluk_4bit = (tahmin_q == y_test).mean()
print(f"Test doğruluğu (4 bit): %{100*dogruluk_4bit:.1f}   (float: %{100*dogruluk_float:.1f})")
print("4 bit ağırlıklar:\n", Wq)
print("Sapmalar (akümülatör başlangıcı):", bq)

## 7. Donanımın yapacağını adım adım taklit et: MAC döngüsü
Aşağıdaki fonksiyon tek bir pikselin puanlarını tam olarak breadboard'daki gibi hesaplar:
akümülatörü sapmayla başlat, 3 kez çarp-biriktir (MAC) yap. Hem MAC sayısını sayıyoruz
hem de akümülatörün ulaştığı en küçük/en büyük değeri izliyoruz, böylece kaç bit gerektiğini görüyoruz.

In [ ]:
mac_sayisi = 0
akum_min, akum_max = 0, 0

def piksel_puanla(r, g, bl):
    global mac_sayisi, akum_min, akum_max
    puan = []
    for k in range(4):                    # 4 renk sınıfı
        akum = int(bq[k])                 # CLRACC + sapma yükle
        for x, w in zip((r, g, bl), Wq[k]):
            akum += x * w                 # MAC komutu
            mac_sayisi += 1
            akum_min, akum_max = min(akum_min, akum), max(akum_max, akum)
        puan.append(akum)                 # RDACC
    return puan

# Döngüyle hesaplanan puanlar matris çarpımıyla aynı mı? (kontrol)
for i in range(min(200, len(Xq_test))):
    assert piksel_puanla(*Xq_test[i]) == list(puanlar[i])
mac_sayisi = 0

for p in Xq_test:
    piksel_puanla(*p)
gerekli_bit = int(np.ceil(np.log2(max(abs(akum_min), akum_max + 1)))) + 1   # +1 işaret biti
print("Piksel başına MAC:", mac_sayisi // len(Xq_test), " -> 5x5 görüntü için:", 25 * mac_sayisi // len(Xq_test))
print("Akümülatör aralığı:", akum_min, "...", akum_max, " -> en az", gerekli_bit, "bit işaretli gerekir")

## 8. Neden 4 bit? Bit sayısı taraması
Aynı deneyi 2'den 8'e kadar her bit sayısı için tekrarlayıp doğruluğu çiziyoruz.
Grafiğin düzleştiği yer, donanımı büyütmeye değmeyeceği noktadır.

In [ ]:
bitler = range(2, 9)
dogruluklar = []
for bit in bitler:
    Wb, bb = nicemle(W, b, bit)
    t = (piksel_nicemle(X_test, bit) @ Wb.T + bb).argmax(axis=1)
    dogruluklar.append(100 * (t == y_test).mean())
    print(f"{bit} bit: %{dogruluklar[-1]:.1f}")

plt.plot(list(bitler), dogruluklar, "o-", label="nicemlenmiş")
plt.axhline(100 * dogruluk_float, ls="--", color="gray", label="float")
plt.xlabel("Piksel ve ağırlık bit sayısı"); plt.ylabel("Test doğruluğu (%)")
plt.legend(); plt.grid(alpha=.3); plt.title("Bit sayısı ve doğruluk")
plt.savefig("bit_taramasi.png", dpi=150); plt.show()

## 9. 5×5 sahne testi: doğru LED yanıyor mu?
Her sahne fotoğrafı 5×5'e küçültülür (her kare, kapladığı alanın ortalama rengi olur).
25 piksel 4 bitlik modelle sınıflandırılır, her renk sayılır, en çok çıkan rengin LED'i yanar.
Eşitlikte eşit olan renklerin hepsi yanar. Sonuç, dosya adındaki gerçek baskın renkle karşılaştırılır.

In [ ]:
dogru_sahne = 0
fig, eksenler = plt.subplots(1, max(len(sahne_dosyalari), 1), figsize=(3 * max(len(sahne_dosyalari), 1), 3), squeeze=False)
for ax, d in zip(eksenler[0], sahne_dosyalari):
    gercek = renk_adi(ad(d).split("_")[-1])
    kucuk = np.asarray(Image.open(d).convert("RGB").resize((5, 5), Image.BOX)).reshape(-1, 3).astype(float)
    sinif = (piksel_nicemle(kucuk, 4) @ Wq.T + bq).argmax(axis=1)
    sayac = np.bincount(sinif, minlength=4)
    yanan = [RENKLER[k] for k in range(4) if sayac[k] == sayac.max()]
    if yanan == [gercek]:
        dogru_sahne += 1
    ax.imshow(kucuk.reshape(5, 5, 3).astype(np.uint8))
    for i, s in enumerate(sinif):
        ax.text(i % 5, i // 5, RENKLER[s][0].upper(), ha="center", va="center", color="white", fontsize=12, weight="bold")
    ax.set_title(f"gerçek: {gercek}\nLED: {', '.join(yanan)}", fontsize=9); ax.axis("off")
    print(f"{os.path.basename(d)}: sayılar {dict(zip(RENKLER, sayac.tolist()))} -> LED: {yanan}")
plt.savefig("sahne_sonuclari.png", dpi=150); plt.show()
print(f"Doğru sahne: {dogru_sahne} / {len(sahne_dosyalari)}")

## 10. Özet (forma bu sayılar girecek)

In [ ]:
ozet = f"""Eğitim pikseli: {len(X_egitim)} ({len(egitim_dosyalari) - 4} görsel), test pikseli: {len(X_test)} (4 görsel)
Test doğruluğu (float): %{100*dogruluk_float:.1f}
Test doğruluğu (4 bit): %{100*dogruluk_4bit:.1f}
Akümülatör aralığı: {akum_min} ... {akum_max} -> en az {gerekli_bit} bit işaretli
5x5 görüntü başına MAC: {25 * 12}
5x5 sahne testi: {dogru_sahne} / {len(sahne_dosyalari)} doğru
4 bit ağırlıklar (R,G,B): {Wq.tolist()}
Sapmalar: {bq.tolist()}"""
print(ozet)
with open("sonuclar.txt", "w") as f:
    f.write(ozet)
try:
    files.download("sonuclar.txt"); files.download("bit_taramasi.png"); files.download("sahne_sonuclari.png")
except Exception:
    pass

## 11. Digital benzetimi için test tablosu üret
Bu hücre, Digital'deki MAC birimini (mac_birimi.dig) test etmek için bir test tablosu yazar.
1. bölüm: 0–15 arası her piksel değeri ile −8…+7 arası her ağırlığın çarpımı (256 durum).
2. bölüm: ilk test sahnesinin 25 pikselinin 4 renk puanı, senin 4 bit ağırlıklarınla (300 MAC).
Beklenen PUAN değerleri yukarıdaki piksel_puanla fonksiyonuyla aynı şekilde hesaplanır.
Dosyanın içeriğini kopyalayıp Digital'deki "Test case" penceresine yapıştır.

In [ ]:
satirlar = ["YUKLE TOPLA B X W CLK PUAN"]
for x in range(16):                       # 1. bölüm: bütün çarpımlar
    for w in range(-8, 8):
        satirlar.append(f"1 0 0 0 0 C 0")
        satirlar.append(f"0 1 0 {x} {w} C {x * w}")

if sahne_dosyalari:                       # 2. bölüm: gerçek bir sahne
    kucuk = np.asarray(Image.open(sahne_dosyalari[0]).convert("RGB").resize((5, 5), Image.BOX)).reshape(-1, 3).astype(float)
    for piksel in piksel_nicemle(kucuk, 4):
        for k in range(4):
            akum = int(bq[k])
            satirlar.append(f"1 0 {akum} 0 0 C {akum}")
            for x, w in zip(piksel, Wq[k]):
                akum += int(x) * int(w)
                satirlar.append(f"0 1 0 {int(x)} {int(w)} C {akum}")

with open("mac_test.txt", "w") as f:
    f.write("\n".join(satirlar) + "\n")
print(len(satirlar) - 1, "test satırı yazıldı. İlk satırlar:")
print("\n".join(satirlar[:6]))
try:
    files.download("mac_test.txt")
except Exception:
    pass